# 🤖 Ciclo — Chatbot com RAG do G.R.U

**G.R.U — Gerenciador de Resíduos Urbanos**

O **Ciclo** é o assistente do Dashboard do Administrador. Ele gera **insights**, faz **previsões** de enchimento das lixeiras e **ajuda na tomada de decisão**, sempre com base nos dados dos gráficos do Dashboard.

Abordagem escolhida: **modelo de LLM por meio de API** - Google Gemini, com a interface do chatbot feita em **Gradio**.

| Tópico pedido | Seção |
|---|---|
| 1. Estrutura do modelo escolhido | [Seção 1](#1) |
| 2. Processo de implementação de RAG | [Seção 3](#3) |
| 3. Explicação do agente implementado | [Seção 4](#4) |
| 4. Código comentado | Todas as células de código |
| Chatbot em Gradio | [Seção 6](#6) |


<a id="1"></a>
## 1. Estrutura do modelo escolhido: Gemini 2.5 Flash (via API)

### 1.1 Por que um LLM via API
- **Não precisamos treinar nem hospedar o modelo**: ele roda na infraestrutura do Google e nós o chamamos por HTTP.
- **Camada gratuita** para estudantes (chave gerada no Google AI Studio).
- **Function calling** (chamada de ferramentas), essencial para o nosso agente consultar os dados do Dashboard em vez de inventar números.
- **API REST nativa** (`generateContent`): chamamos o Gemini direto por HTTP, com a chave no cabeçalho `x-goog-api-key`. Esse é o formato que aceita as chaves novas do AI Studio (que começam com `AQ.`).

### 1.2 Arquitetura do modelo
O Gemini é um **LLM baseado em Transformer** (o Google não publica todos os detalhes internos da versão Flash). O funcionamento geral de um LLM desse tipo:

```
Texto de entrada ──► Tokenizador ──► Embeddings dos tokens (+ posição)
                                         │
                                         ▼
                ┌─────────── N blocos Transformer ───────────┐
                │  Autoatenção (cada token "olha" os outros)  │
                │  Rede feed-forward                          │
                │  Normalização + conexões residuais          │
                └─────────────────────────────────────────────┘
                                         │
                                         ▼
                 Distribuição de probabilidade do próximo token
                                         │
                                         ▼
          Amostragem (temperatura) ──► próximo token ──► repete (autorregressivo)
```

- **Tokenização**: o texto vira pedaços (tokens); o modelo prevê um token por vez.
- **Autoatenção**: permite relacionar partes distantes do texto, por exemplo a pergunta do usuário com um trecho recuperado pelo RAG.
- **Janela de contexto**: quantos tokens cabem numa chamada (prompt + resposta). É onde colocamos as instruções, o histórico, os trechos do RAG e os resultados das ferramentas.
- **Temperatura**: controla a aleatoriedade. Usamos **0.3** para respostas mais estáveis e factuais.
- **Function calling**: em vez de texto, o modelo pode responder com um pedido estruturado (`nome_da_ferramenta` + argumentos em JSON). O nosso código executa e devolve o resultado.

### 1.3 Limitações que o projeto contorna
| Limitação do LLM | Como o Ciclo contorna |
|---|---|
| Não conhece os dados do G.R.U | **RAG** + **ferramentas** sobre os dados do Dashboard |
| Pode "alucinar" números | Prompt proíbe inventar dados; números vêm das ferramentas |
| Não faz contas de forma confiável | A previsão é calculada em código (ferramenta `prever_enchimento`) |
| Precisa de internet e de chave | **Modo offline** com as mesmas ferramentas e o mesmo RAG |
| Sobrecarga do serviço (erro 503) | Novas tentativas com espera e **modelos reserva** |

## 2. Configuração

Instalação das bibliotecas. No Google Colab ou num ambiente novo, execute a célula abaixo uma vez.

In [ ]:
# Instala as dependências (execute uma vez).
#  - gradio: interface web do chatbot
#  - requests: chamadas HTTP à API do Gemini
#  - scikit-learn: TF-IDF e similaridade de cosseno do RAG
#  - matplotlib: gráficos do Dashboard dentro da interface
%pip install -q "gradio>=6" requests scikit-learn matplotlib

In [ ]:
import os
import json
from datetime import datetime, timedelta, timezone

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Configuração do LLM ──────────────────────────────────────────────────────
# A chave NUNCA deve ficar escrita no notebook (ele vai para o GitHub).
# Defina a variável de ambiente CICLO_API_KEY antes de abrir o notebook, ou
# descomente as duas linhas abaixo para digitá-la de forma oculta.
# from getpass import getpass
# os.environ["CICLO_API_KEY"] = getpass("Chave da API do Gemini: ")

API_KEY  = os.environ.get("CICLO_API_KEY", "")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta"   # API nativa do Gemini
# Um ou mais modelos separados por vírgula: o primeiro é o principal e os
# demais são reservas, usados se o anterior estiver sobrecarregado.
import re

def limpar_nome_do_modelo(nome):
    # Aceita aspas, espaços, caracteres invisíveis e o prefixo "models/" do AI Studio.
    nome = re.sub(r"[\u200b-\u200d\ufeff\"'`]", "", nome).strip()
    return nome.removeprefix("models/")

MODELOS  = [m for m in (limpar_nome_do_modelo(x) for x in os.environ.get("CICLO_MODEL", "gemini-2.5-flash").split(",")) if m]
print("Modelos:", MODELOS)

MODO = "llm" if API_KEY else "offline"
print(f"Modo do Ciclo: {MODO}" + ("" if API_KEY else "  (defina CICLO_API_KEY para usar o Gemini)"))

### 2.1 Dados do Dashboard (snapshot)

O Ciclo responde sobre os **mesmos dados que alimentam os gráficos** do Dashboard. O app e o site montam esse "snapshot" e o enviam junto com cada pergunta. Aqui usamos um snapshot de exemplo com a mesma estrutura.

| Chave do snapshot | Gráfico do Dashboard |
|---|---|
| `kpis` | Cartões do topo (Lixeiras, Ocupação média, Para coletar, Coletas 30 dias) |
| `lixeiras[].ocupacao` | Barras "Ocupação por lixeira" |
| `composicaoGeral` | Rosca "Tipos de resíduo" |
| `ranking` | "Ranking de coletores" |
| `atividades` | "Atividades recentes" |

In [ ]:
AGORA = datetime(2026, 9, 29, 12, 0, tzinfo=timezone.utc)

def ha(dias=0, horas=0):
    # Data ISO de 'dias'/'horas' atrás em relação a AGORA.
    return (AGORA - timedelta(days=dias, hours=horas)).isoformat()

SNAPSHOT = {
    "instituicao": "Prefeitura Municipal",
    "geradoEm": AGORA.isoformat(),
    "kpis": {"lixeiras": 5, "ocupacaoMedia": 54, "paraColetar": 2, "coletas30d": 12},
    "lixeiras": [
        {"nome": "Lixeira Praça Central",        "endereco": "Praça da Sé, 100",        "ocupacao": 58,
         "status": "Ocupação média", "composicao": {"Plástico": 45, "Papel": 30, "Vidro": 15, "Metal": 10}, "ultimaColeta": ha(dias=2)},
        {"nome": "Lixeira Parque das Águas",     "endereco": "Rua das Águas, 230",      "ocupacao": 81,
         "status": "Ocupação alta",  "composicao": {"Plástico": 55, "Papel": 20, "Metal": 15, "Vidro": 10}, "ultimaColeta": ha(dias=3)},
        {"nome": "Lixeira Biblioteca Municipal", "endereco": "Rua da Consolação, 42",   "ocupacao": 37,
         "status": "Ocupação baixa", "composicao": {"Papel": 62, "Plástico": 28, "Metal": 6, "Vidro": 4},   "ultimaColeta": ha(dias=1)},
        {"nome": "Lixeira Feira Livre",          "endereco": "Rua Augusta, 1200",       "ocupacao": 92,
         "status": "Cheia",          "composicao": {"Metal": 40, "Plástico": 30, "Vidro": 20, "Papel": 10}, "ultimaColeta": ha(dias=2)},
        {"nome": "Lixeira Campus Norte",         "endereco": "Av. Universitária, 500",  "ocupacao": 5,
         "status": "Vazia",          "composicao": {"Vidro": 50, "Metal": 25, "Plástico": 15, "Papel": 10}, "ultimaColeta": ha(horas=3)},
    ],
    "composicaoGeral": {"Plástico": 41, "Papel": 23, "Metal": 22, "Vidro": 14},
    "ranking": [{"nome": "João Silva", "coletas": 7}, {"nome": "Maria Souza", "coletas": 4}, {"nome": "Pedro Lima", "coletas": 1}],
    "atividades": [
        {"coletor": "João Silva",  "lixeira": "Lixeira Campus Norte",  "dataHora": ha(horas=3), "ocupacao": 95},
        {"coletor": "Maria Souza", "lixeira": "Lixeira Biblioteca Municipal", "dataHora": ha(dias=1), "ocupacao": 48},
    ],
}
print(f"{len(SNAPSHOT['lixeiras'])} lixeiras no snapshot de {SNAPSHOT['instituicao']}")

<a id="3"></a>
## 3. Processo de implementação de RAG

**RAG (Retrieval-Augmented Generation)**: antes de o LLM responder, **recuperamos** (Retrieval) trechos relevantes de uma base de conhecimento própria e os colocamos no prompt, **aumentando** (Augmented) o contexto da **geração** (Generation). Assim o modelo responde com base no conhecimento do G.R.U, e não só no que aprendeu no treinamento.

```
           INDEXAÇÃO (uma vez)                          CONSULTA (a cada pergunta)
┌──────────────────────────────────┐        ┌──────────────────────────────────────────┐
│ 1. Documentos .md do G.R.U       │        │ 5. Pergunta do usuário                   │
│ 2. Chunking (um pedaço por seção)│        │ 6. Vetorizar a pergunta (mesmo TF-IDF)   │
│ 3. Vetorização TF-IDF            │──────► │ 7. Similaridade de cosseno com os chunks │
│ 4. Matriz de vetores (índice)    │        │ 8. Top-K trechos mais parecidos          │
└──────────────────────────────────┘        │ 9. Prompt = instruções + trechos + dados │
                                            │10. LLM gera a resposta                   │
                                            └──────────────────────────────────────────┘
```

**Por que TF-IDF?** É leve, roda offline, não custa chamadas de API e é fácil de explicar. Para uma base pequena e com vocabulário controlado (termos do próprio G.R.U), funciona bem. A evolução natural seria trocar o passo 3 por **embeddings semânticos** (que captam sinônimos) e um banco vetorial; o resto do pipeline continua igual.

### 3.1 Base de conhecimento
Os mesmos textos do backend (`backend/src/data/ciclo-knowledge/`): conceitos do G.R.U, como ler cada gráfico, boas práticas de coleta e a descrição do próprio Ciclo.

In [ ]:
# Base de conhecimento do Ciclo: mesmos arquivos de backend/src/data/ciclo-knowledge/
# (copiados aqui para o notebook funcionar sozinho, por exemplo no Colab).
BASE_CONHECIMENTO = {
    '01-conceitos.md': '# Conceitos do G.R.U\n\nO G.R.U (Gerenciador de Resíduos Urbanos) monitora lixeiras inteligentes\nespalhadas por instituições (prefeituras, universidades, terminais). Cada\nlixeira tem sensores que estimam a **ocupação** (percentual da capacidade em\nuso) e a **composição** do lixo por tipo de material.\n\n## Status de ocupação\n\n- **Vazia** (0% a 9%): acabou de ser coletada ou tem uso muito baixo.\n- **Ocupação baixa** (10% a 39%): tranquila, sem urgência de coleta.\n- **Ocupação média** (40% a 69%): merece acompanhamento, mas ainda não é prioridade.\n- **Ocupação alta** (70% a 89%): entra na fila de coleta em breve.\n- **Cheia** (90% a 100%): prioridade máxima, risco de transbordar.\n\nLixeiras com status "alta" ou "cheia" são consideradas **prioritárias** e é\nisso que o KPI "Para coletar" do dashboard soma.\n\n## Tipos de resíduo\n\nO G.R.U rastreia quatro categorias: **Plástico**, **Vidro**, **Metal** e\n**Papel**. O "material predominante" de uma lixeira é o tipo com maior\npercentual na composição lida pelos sensores.\n\n## Coletores e instituições\n\nUm **Coletor** é o usuário que esvazia as lixeiras fisicamente. Ele só\nenxerga (e só pode coletar) as lixeiras das instituições às quais o\nAdministrador o vinculou. Cada coleta registrada vira uma **visita** no\nhistórico: guarda quando, em qual lixeira, com qual ocupação e qual material\npredominante no momento da coleta.',
    '02-dashboard.md': '# Como ler os gráficos do Dashboard\n\n## KPIs no topo\n\n- **Lixeiras**: quantas lixeiras a instituição tem cadastradas.\n- **Ocupação média**: média simples da ocupação (%) de todas as lixeiras\n  agora. Acima de 70% é sinal de que a operação está atrasada.\n- **Para coletar**: quantas lixeiras estão em status "alta" ou "cheia" —\n  é a fila de prioridade de hoje.\n- **Coletas (30 dias)**: quantas coletas (visitas) foram registradas nos\n  últimos 30 dias, em todas as lixeiras da instituição.\n\n## Ocupação por lixeira (barras)\n\nUma barra por lixeira, ordenada da mais cheia para a mais vazia. Serve para\ndecidir a rota de coleta do dia: comece pelas barras mais altas.\n\n## Tipos de resíduo (rosca)\n\nMostra a composição agregada de material em todas as lixeiras, ponderada\npela ocupação de cada uma (uma lixeira cheia pesa mais no gráfico do que uma\nquase vazia). Ajuda a decidir se vale reforçar a separação de um material\nespecífico ou negociar um parceiro de reciclagem para ele.\n\n## Ranking de coletores\n\nNúmero de coletas que cada coletor da instituição já realizou. Serve para\nver quem está mais ativo e se a carga está bem distribuída entre a equipe.\n\n## Atividades recentes\n\nAs últimas coletas registradas (quem coletou, onde e quando), em ordem do\nmais recente para o mais antigo.',
    '03-boas-praticas.md': '# Boas práticas de coleta e gestão\n\n## Priorização de rota\n\nA forma mais eficiente de planejar o dia é olhar o gráfico "Ocupação por\nlixeira" e visitar primeiro as prioritárias (alta/cheia), agrupando por\nproximidade geográfica quando possível, para economizar deslocamento.\n\n## Ocupação média alta e persistente\n\nSe a ocupação média fica repetidamente acima de 70%, é sinal de que a\nfrequência de coleta está menor do que a geração de lixo daquele local.\nAlternativas: aumentar a frequência de coleta, adicionar mais lixeiras no\nlocal, ou entender se houve um evento pontual (feira, evento) que gerou pico.\n\n## Poucas coletas de um coletor\n\nUm coletor com número de coletas muito abaixo da média do ranking pode\nindicar rota mal distribuída, poucas lixeiras vinculadas a ele, ou um\nproblema operacional a conversar com a pessoa.\n\n## Composição de material concentrada\n\nQuando um material domina a composição (por exemplo, mais de 60% de\nplástico), vale avaliar parcerias específicas de reciclagem para esse\nmaterial, ou lixeiras dedicadas (coleta seletiva) naquele ponto.\n\n## Lixeiras sem coleta recente\n\nUma lixeira com ocupação subindo e sem visita registrada há muitos dias é\num ponto cego da operação: vale checar se algum coletor está vinculado a\nela e se a rota está cobrindo aquele endereço.',
    '04-ciclo.md': '# Sobre o Ciclo\n\nCiclo é o assistente virtual do G.R.U. Ele ajuda o Administrador a\ninterpretar os gráficos do Dashboard, aponta lixeiras e coletores que\nprecisam de atenção, e sugere ações com base nos dados mostrados na tela\nnaquele momento (KPIs, ocupação por lixeira, composição de material,\nranking de coletores e atividades recentes).\n\nCiclo não inventa números: quando responde sobre uma lixeira, um coletor ou\numa métrica específica, ele se baseia nos dados reais enviados pelo\naplicativo junto da pergunta. Quando o dado pedido não estiver disponível\nno que foi enviado, Ciclo avisa isso em vez de adivinhar.',
}
print('Documentos:', list(BASE_CONHECIMENTO))

### 3.2 Chunking
Cada documento é dividido por seção (`## Título`). Chunks pequenos deixam a recuperação mais precisa: trazemos só o trecho que interessa, não o documento inteiro. O título do documento é repetido em cada chunk para não perder o contexto.

In [ ]:
import re

def fazer_chunks(base):
    # Divide cada documento em pedaços, um por seção '## ...'.
    chunks = []
    for fonte, texto in base.items():
        titulo_doc = re.search(r"^#\s+(.+)$", texto, re.M).group(1)
        # Quebra antes de cada linha que começa com '## '
        for secao in re.split(r"\n(?=##\s)", texto):
            secao = secao.strip()
            if not secao or secao.startswith("# "):   # pula o título solto do arquivo
                continue
            chunks.append({"fonte": fonte, "texto": f"{titulo_doc}\n{secao}"})
    return chunks

CHUNKS = fazer_chunks(BASE_CONHECIMENTO)
print(f"{len(CHUNKS)} chunks gerados. Exemplo:\n")
print(CHUNKS[0]["texto"][:300], "...")

### 3.3 Vetorização (TF-IDF) e índice
**TF-IDF** dá peso alto a palavras frequentes num chunk e raras no resto da base (as que realmente caracterizam o trecho). Removemos acentos e *stopwords* (palavras sem significado próprio, como "de", "que").

In [ ]:
import unicodedata

STOPWORDS = """a o as os de da do das dos e ou que com em no na nos nas um uma uns umas
ao aos se por para como mais menos seu sua seus suas ele ela eles elas ser esta
estao foi sao isso esse essa este num numa quando cada ja nao sim""".split()

def normalizar(texto):
    # Minúsculas e sem acentos: 'Ocupação' e 'ocupacao' viram o mesmo termo.
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

# O vetorizador aprende o vocabulário e os pesos IDF a partir dos chunks.
vetorizador = TfidfVectorizer(
    preprocessor=normalizar,
    stop_words=STOPWORDS,
    token_pattern=r"(?u)\b[a-z0-9]{3,}\b",   # só palavras com 3+ caracteres
)
MATRIZ_CHUNKS = vetorizador.fit_transform([c["texto"] for c in CHUNKS])
print("Índice:", MATRIZ_CHUNKS.shape[0], "chunks x", MATRIZ_CHUNKS.shape[1], "termos")

### 3.4 Recuperação (Retrieval)
A pergunta é vetorizada com o **mesmo** vetorizador e comparada com todos os chunks pela **similaridade de cosseno** (1 = idênticos, 0 = nada em comum). Devolvemos os `top_k` mais parecidos.

In [ ]:
def buscar_conhecimento(consulta, top_k=3):
    # Etapa 'R' do RAG: devolve os trechos mais relevantes para a consulta.
    vetor = vetorizador.transform([consulta])
    scores = cosine_similarity(vetor, MATRIZ_CHUNKS)[0]
    melhores = np.argsort(scores)[::-1][:top_k]
    return [
        {"fonte": CHUNKS[i]["fonte"], "score": round(float(scores[i]), 3), "texto": CHUNKS[i]["texto"]}
        for i in melhores if scores[i] > 0
    ]

# Teste: a pergunta usa palavras que aparecem na seção de status de ocupação.
for r in buscar_conhecimento("o que significa ocupação alta?"):
    print(f"[{r['score']}] {r['fonte']}: {r['texto'].splitlines()[1]}")

### 3.5 Aumento do prompt (Augmented)
Os trechos recuperados entram no **prompt de sistema**, junto das regras de comportamento do Ciclo. O LLM passa a ter, na própria janela de contexto, o conhecimento específico do G.R.U.

In [ ]:
def prompt_de_sistema(snapshot, trechos):
    contexto = "\n\n".join(f"[{i+1}] ({t['fonte']})\n{t['texto']}" for i, t in enumerate(trechos)) \
               or "(nenhum trecho relevante encontrado)"
    return f'''Você é o Ciclo, assistente virtual do G.R.U (Gerenciador de Resíduos Urbanos).
Você ajuda o Administrador de "{snapshot['instituicao']}" a entender o Dashboard, gerar insights, fazer previsões e decidir ações.

Regras:
- Responda em português do Brasil, de forma curta e direta (até ~6 frases ou uma lista curta).
- Use as FERRAMENTAS para qualquer número sobre lixeiras, coletores, material ou previsões. Nunca invente dados.
- Previsões são estimativas lineares (supõem ritmo constante): deixe isso claro quando der uma data.
- Quando fizer sentido, termine com uma ação recomendada.
- Se o dado pedido não existir no dashboard, diga que não tem essa informação.

Base de conhecimento recuperada (RAG):
{contexto}'''

print(prompt_de_sistema(SNAPSHOT, buscar_conhecimento("como ler o gráfico de rosca"))[:900], "...")

<a id="4"></a>
## 4. O agente implementado

O Ciclo é um **agente com ferramentas** (padrão **ReAct**: *Reason + Act*). Um chatbot comum só gera texto; o agente pode **decidir executar ações** e usar o resultado para responder.

```
 Pergunta ──► [RAG] ──► LLM ──┬── responde em texto ────────────────► Resposta final
                              │
                              └── pede ferramenta(s) ──► código executa sobre o snapshot
                                           ▲                              │
                                           └──── resultado (JSON) ◄───────┘
                                     (repete até 4 voltas)
```

### 4.1 Componentes
| Componente | Papel |
|---|---|
| **LLM (Gemini)** | Raciocina e decide *qual* ferramenta usar e com quais argumentos |
| **RAG** | Traz conhecimento de domínio (conceitos, gráficos, boas práticas) |
| **Ferramentas** | Funções Python que leem os dados do Dashboard e fazem cálculos |
| **Memória** | Histórico das últimas mensagens enviado a cada chamada |
| **Loop de controle** | Executa as ferramentas pedidas e limita o número de voltas |

### 4.2 Ferramentas e correlação com os gráficos
Cada ferramenta está ligada a um gráfico do Dashboard. Na resposta, o Ciclo informa quais gráficos usou, e a interface mostra/destaca esse gráfico. É essa a **correlação entre o chatbot e o Dashboard**.

| Ferramenta | O que faz | Gráfico |
|---|---|---|
| `resumo_dashboard` | KPIs gerais | `kpis` |
| `lixeiras_prioritarias` | Lixeiras acima de X%, da mais cheia para a mais vazia | `ocupacao` |
| `detalhes_lixeira` | Ocupação, composição e última coleta de uma lixeira | `ocupacao` |
| `prever_enchimento` | **Previsão** de quando cada lixeira chega a 90% | `ocupacao` |
| `composicao_residuos` | Composição geral por material | `residuos` |
| `desempenho_coletores` | Ranking e quem está abaixo da média | `ranking` |
| `atividades_recentes` | Últimas coletas | `atividades` |
| `buscar_conhecimento` | Busca na base do RAG (o agente pode pedir mais contexto) | — |

### 4.3 Como a previsão é calculada
Após cada coleta a lixeira volta a 0%. A taxa média de enchimento é:

$$\text{taxa (\%/dia)} = \frac{\text{ocupação atual}}{\text{dias desde a última coleta}} \qquad \text{dias até encher} = \frac{90\% - \text{ocupação atual}}{\text{taxa}}$$

É uma **regressão linear simples** com dois pontos (0% na coleta e a ocupação atual). Supõe ritmo constante, então o Ciclo sempre avisa que é uma estimativa. O cálculo é feito **em código**, não pelo LLM, para evitar erros de conta.

In [ ]:
LIMITE_CHEIA = 90   # % a partir do qual consideramos a lixeira cheia

def _resumo(l):
    return {k: l[k] for k in ("nome", "endereco", "ocupacao", "status")}

def _achar_lixeira(snap, nome):
    # Aceita o nome completo ou só um pedaço ('Feira' encontra 'Lixeira Feira Livre').
    alvo = (nome or "").lower()
    return next((l for l in snap["lixeiras"] if alvo and alvo in l["nome"].lower()), None)

def _prazo_em_texto(dias):
    if dias < 1:
        h = max(1, round(dias * 24))
        return f"~{h} hora" + ("s" if h > 1 else "")
    d = round(dias)
    return f"~{d} dia" + ("s" if d > 1 else "")

def _prever_lixeira(l, agora):
    # Previsão linear de enchimento de UMA lixeira (explicação na seção 4.3).
    if l["ocupacao"] >= LIMITE_CHEIA:
        return {**_resumo(l), "diasParaEncher": 0, "previsao": "Já está cheia: coletar hoje."}
    if not l.get("ultimaColeta"):
        return {**_resumo(l), "diasParaEncher": None, "previsao": "Sem histórico de coleta para estimar."}
    dias = (agora - datetime.fromisoformat(l["ultimaColeta"])).total_seconds() / 86400
    if dias < 0.25 or l["ocupacao"] <= 0:
        return {**_resumo(l), "diasParaEncher": None, "previsao": "Coletada há pouco; ainda não dá para estimar."}
    taxa = l["ocupacao"] / dias                          # %/dia
    restantes = (LIMITE_CHEIA - l["ocupacao"]) / taxa    # dias até 90%
    return {**_resumo(l), "taxaPorDia": round(taxa, 1), "diasParaEncher": round(restantes, 1),
            "emTexto": _prazo_em_texto(restantes),
            "dataEstimada": (agora + timedelta(days=restantes)).date().isoformat()}

# ── As ferramentas: cada uma recebe o snapshot + argumentos e devolve um dict/list ──
def t_resumo_dashboard(snap):
    return {"instituicao": snap["instituicao"], **snap["kpis"]}

def t_lixeiras_prioritarias(snap, minimo=70):
    lista = sorted(snap["lixeiras"], key=lambda l: -l["ocupacao"])
    return [_resumo(l) for l in lista if l["ocupacao"] >= minimo]

def t_detalhes_lixeira(snap, nome):
    return _achar_lixeira(snap, nome) or {"erro": f"Lixeira '{nome}' não encontrada."}

def t_prever_enchimento(snap, nome=None):
    agora = datetime.fromisoformat(snap["geradoEm"])
    if nome:
        l = _achar_lixeira(snap, nome)
        return _prever_lixeira(l, agora) if l else {"erro": f"Lixeira '{nome}' não encontrada."}
    prev = [_prever_lixeira(l, agora) for l in snap["lixeiras"]]
    return sorted(prev, key=lambda p: 999 if p["diasParaEncher"] is None else p["diasParaEncher"])

def t_composicao_residuos(snap):
    itens = sorted(snap["composicaoGeral"].items(), key=lambda kv: -kv[1])
    return {"composicao": dict(itens), "predominante": itens[0][0] if itens else None}

def t_desempenho_coletores(snap):
    r = snap["ranking"]
    media = sum(c["coletas"] for c in r) / len(r) if r else 0
    return {"media": round(media, 1),
            "ranking": [{**c, "abaixoDaMedia": c["coletas"] < media * 0.6} for c in r]}

def t_atividades_recentes(snap, limite=5):
    return snap["atividades"][:limite]

def t_buscar_conhecimento(snap, consulta):
    return buscar_conhecimento(consulta, 3)

print("Teste da previsão:")
for p in t_prever_enchimento(SNAPSHOT):
    print(" •", p["nome"], "→", p.get("emTexto") or p.get("previsao"))

### 4.4 Registro das ferramentas (function calling)
O LLM não vê o código Python: ele vê **o nome, a descrição e os parâmetros** de cada ferramenta (JSON Schema). É com base na descrição que ele decide quando usar cada uma. Por isso as descrições são escritas com cuidado.

In [ ]:
def _params(props=None, obrigatorios=None):
    return {"type": "object", "properties": props or {}, **({"required": obrigatorios} if obrigatorios else {})}

# nome → (função, descrição para o LLM, parâmetros, gráfico relacionado)
FERRAMENTAS = {
    "resumo_dashboard": (t_resumo_dashboard,
        "Resumo geral: KPIs (lixeiras, ocupação média, para coletar, coletas em 30 dias).",
        _params(), "kpis"),
    "lixeiras_prioritarias": (t_lixeiras_prioritarias,
        "Lixeiras da mais cheia para a mais vazia (gráfico 'Ocupação por lixeira'). Use para planejar a rota.",
        _params({"minimo": {"type": "number", "description": "Ocupação mínima (%). Padrão 70."}}), "ocupacao"),
    "detalhes_lixeira": (t_detalhes_lixeira,
        "Detalhes de uma lixeira: ocupação, composição de material e última coleta.",
        _params({"nome": {"type": "string", "description": "Nome ou parte do nome."}}, ["nome"]), "ocupacao"),
    "prever_enchimento": (t_prever_enchimento,
        "Previsão de quando as lixeiras vão encher (>=90%). Sem 'nome', devolve todas, das mais urgentes para as menos.",
        _params({"nome": {"type": "string", "description": "Opcional: só esta lixeira."}}), "ocupacao"),
    "composicao_residuos": (t_composicao_residuos,
        "Composição geral por tipo de material (gráfico de rosca 'Tipos de resíduo').",
        _params(), "residuos"),
    "desempenho_coletores": (t_desempenho_coletores,
        "Ranking de coletores por número de coletas, com a média da equipe.",
        _params(), "ranking"),
    "atividades_recentes": (t_atividades_recentes,
        "Últimas coletas registradas (quem, onde e quando).",
        _params({"limite": {"type": "number", "description": "Quantas trazer. Padrão 5."}}), "atividades"),
    "buscar_conhecimento": (t_buscar_conhecimento,
        "Busca na base de conhecimento do G.R.U (conceitos, como ler os gráficos, boas práticas). Use para perguntas conceituais.",
        _params({"consulta": {"type": "string", "description": "O que buscar."}}, ["consulta"]), None),
}

# Formato 'tools' aceito pela API (compatível com OpenAI)
SCHEMAS = [{"type": "function", "function": {"name": n, "description": d, "parameters": p}}
           for n, (_, d, p, _g) in FERRAMENTAS.items()]

def executar_ferramenta(nome, args, snap):
    # Executa com segurança: nome inválido ou argumento errado viram um erro legível para o LLM.
    if nome not in FERRAMENTAS:
        return {"erro": f"Ferramenta desconhecida: {nome}"}
    try:
        return FERRAMENTAS[nome][0](snap, **(args or {}))
    except Exception as e:
        return {"erro": f"Falha em {nome}: {e}"}

print(len(SCHEMAS), "ferramentas registradas:", ", ".join(FERRAMENTAS))

### 4.5 O loop do agente
Esta é a parte central. A cada volta, o LLM ou responde em texto (fim) ou pede ferramentas (`functionCall`); o código executa, devolve os resultados como `functionResponse` e chama o LLM de novo.

Formato da API nativa do Gemini: as instruções vão em `systemInstruction`, a conversa em `contents` (papéis `user` e `model`) e as ferramentas em `tools.functionDeclarations`.

In [ ]:
import requests

MAX_PASSOS = 4   # limite de voltas: evita loops infinitos e custo alto

def _declaracoes():
    # Ferramentas no formato do Gemini. Ferramentas sem parâmetros não podem
    # ter "properties" vazio, então nesses casos omitimos "parameters".
    decl = []
    for s in SCHEMAS:
        f = s["function"]
        d = {"name": f["name"], "description": f["description"]}
        if f["parameters"].get("properties"):
            d["parameters"] = f["parameters"]
        decl.append(d)
    return decl

import time

STATUS_TEMPORARIOS = {429, 500, 502, 503, 504}   # sobrecarga/limite: vale tentar de novo
ESPERAS = [1, 3]                                 # segundos antes da 2ª e da 3ª tentativa

def _chamar_gemini(sistema, contents, com_ferramentas=True):
    # Uma chamada à API nativa (generateContent). A chave vai no cabeçalho.
    corpo = {
        "systemInstruction": {"parts": [{"text": sistema}]},
        "contents": contents,
        "generationConfig": {"temperature": 0.3},
    }
    if com_ferramentas:
        corpo["tools"] = [{"functionDeclarations": _declaracoes()}]

    ultimo_erro = None
    for modelo in MODELOS:                       # principal e, se preciso, as reservas
        for tentativa in range(len(ESPERAS) + 1):
            r = requests.post(f"{BASE_URL}/models/{modelo}:generateContent",
                              headers={"x-goog-api-key": API_KEY}, json=corpo, timeout=60)
            if r.ok:
                return r.json()["candidates"][0]["content"]
            ultimo_erro = RuntimeError(f"Gemini ({modelo}) respondeu {r.status_code}: {r.text[:300]}")
            if r.status_code in STATUS_TEMPORARIOS and tentativa < len(ESPERAS):
                print(f"{r.status_code} no {modelo}; tentando de novo em {ESPERAS[tentativa]}s...")
                time.sleep(ESPERAS[tentativa])
                continue
            break                                # erro definitivo ou tentativas esgotadas
        if r.status_code not in STATUS_TEMPORARIOS | {404}:
            raise ultimo_erro                    # chave inválida etc.: trocar de modelo não resolve
    raise ultimo_erro

def perguntar_llm(pergunta, snap, historico=None):
    trechos = buscar_conhecimento(pergunta, 3)                       # 1. Retrieval
    sistema = prompt_de_sistema(snap, trechos)                       # 2. Augmented

    # Memória curta: papéis "user" e "model" (o Gemini chama o assistente de "model").
    contents = [{"role": "model" if m["role"] == "assistant" else "user",
                 "parts": [{"text": m["content"]}]} for m in (historico or [])[-8:]]
    contents.append({"role": "user", "parts": [{"text": pergunta}]})

    usadas = []
    for _ in range(MAX_PASSOS):
        conteudo = _chamar_gemini(sistema, contents)                 # 3. Generation
        partes = conteudo.get("parts", [])
        chamadas = [p["functionCall"] for p in partes if "functionCall" in p]

        if not chamadas:                                             # respondeu em texto: fim
            texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
            return finalizar(texto, usadas, trechos, "llm")

        contents.append(conteudo)          # devolve o pedido inteiro (preserva thoughtSignature)
        respostas = []
        for c in chamadas:                                           # 4. Ação
            resultado = executar_ferramenta(c["name"], c.get("args", {}), snap)
            usadas.append(c["name"])
            # "response" precisa ser um objeto; listas vão dentro de "resultado".
            respostas.append({"functionResponse": {"name": c["name"], "response": {"resultado": resultado}}})
        contents.append({"role": "user", "parts": respostas})

    # Estourou o limite: pede uma resposta final com o que já tem.
    contents.append({"role": "user", "parts": [{"text": "Responda agora com o que já tem, sem chamar ferramentas."}]})
    conteudo = _chamar_gemini(sistema, contents, com_ferramentas=False)
    texto = "".join(p.get("text", "") for p in conteudo.get("parts", []) if not p.get("thought"))
    return finalizar(texto, usadas, trechos, "llm")

def finalizar(texto, usadas, trechos, modo):
    graficos = list(dict.fromkeys(FERRAMENTAS[n][3] for n in usadas if FERRAMENTAS.get(n, (0,0,0,None))[3]))
    return {"resposta": texto.strip() or "Não consegui formular uma resposta agora.",
            "graficos": graficos, "ferramentas": list(dict.fromkeys(usadas)),
            "fontes": list(dict.fromkeys(t["fonte"] for t in trechos)), "modo": modo}

### 4.6 Modo offline (sem chave de API)
Para demonstrar sem internet ou sem chave, um roteador por **palavras-chave** escolhe a ferramenta no lugar do LLM e o texto é montado em código. As ferramentas e o RAG são os mesmos; só a "inteligência" da escolha e da redação é mais simples.

In [ ]:
def perguntar_offline(pergunta, snap):
    p = normalizar(pergunta)
    tem = lambda *palavras: any(w in p for w in palavras)
    trechos = buscar_conhecimento(pergunta, 3)
    citada = next((l for l in snap["lixeiras"] if normalizar(l["nome"].replace("Lixeira ", "")) in p), None)

    if tem("prev", "quando", "encher", "enche", "semana", "amanha"):
        f = "prever_enchimento"
        r = executar_ferramenta(f, {"nome": citada["nome"]} if citada else {}, snap)
        lista = [x for x in (r if isinstance(r, list) else [r]) if x.get("diasParaEncher") is not None][:4]
        texto = "Previsão de enchimento (estimativa linear, supondo o ritmo atual):\n" + "\n".join(
            f"• {x['nome']}: já está cheia ({x['ocupacao']}%), coletar hoje." if x["diasParaEncher"] == 0
            else f"• {x['nome']}: {x['ocupacao']}% hoje, deve encher em {x['emTexto']} ({x['dataEstimada']})."
            for x in lista)
    elif citada:
        f = "detalhes_lixeira"
        mat = max(citada["composicao"].items(), key=lambda kv: kv[1])
        texto = f"{citada['nome']} está com {citada['ocupacao']}% ({citada['status']}), predominância de {mat[0]} ({mat[1]}%)."
    elif tem("coletor", "equipe", "ranking", "desempenho"):
        f = "desempenho_coletores"
        r = executar_ferramenta(f, {}, snap)
        baixos = [c["nome"] for c in r["ranking"] if c["abaixoDaMedia"]]
        texto = (f"Média de {r['media']} coletas por coletor; {r['ranking'][0]['nome']} lidera com {r['ranking'][0]['coletas']}. "
                 + (f"Abaixo da média: {', '.join(baixos)}. Vale revisar a rota." if baixos else "Carga bem distribuída."))
    elif tem("material", "residuo", "plastico", "vidro", "metal", "papel", "composi", "rosca"):
        f = "composicao_residuos"
        r = executar_ferramenta(f, {}, snap)
        texto = ("Composição geral: " + ", ".join(f"{k} {v}%" for k, v in r["composicao"].items())
                 + f". {r['predominante']} predomina: avalie coleta seletiva ou parceria de reciclagem.")
    elif tem("rota", "prioridad", "coletar", "cheia", "urgente", "primeiro"):
        f = "lixeiras_prioritarias"
        r = executar_ferramenta(f, {"minimo": 70}, snap)
        texto = ("Prioridade de coleta hoje:\n" + "\n".join(f"{i+1}. {l['nome']}: {l['ocupacao']}%" for i, l in enumerate(r))
                 if r else "Nenhuma lixeira acima de 70% agora.")
    elif tem("atividade", "ultimas", "recente", "aconteceu", "historico"):
        f = "atividades_recentes"
        r = executar_ferramenta(f, {"limite": 5}, snap)
        texto = ("Últimas atividades:\n" + "\n".join(f"• {a['coletor']} coletou {a['lixeira']} ({a['ocupacao']}% cheia)" for a in r)
                 if r else "Nenhuma atividade registrada ainda.")
    elif tem("resumo", "geral", "situacao", "insight", "visao"):
        f = "resumo_dashboard"
        k = snap["kpis"]
        texto = (f"{snap['instituicao']}: {k['lixeiras']} lixeiras, ocupação média de {k['ocupacaoMedia']}%, "
                 f"{k['paraColetar']} para coletar e {k['coletas30d']} coletas em 30 dias.")
    elif trechos:
        f = "buscar_conhecimento"
        texto = "\n".join(trechos[0]["texto"].splitlines()[1:]).lstrip("# ").strip()
    else:
        f = None
        texto = "Posso ajudar com: resumo, prioridade de coleta, previsão de enchimento, materiais e coletores."
    return finalizar(texto, [f] if f else [], trechos, "offline")

def perguntar_ao_ciclo(pergunta, snap=SNAPSHOT, historico=None):
    # Ponto de entrada: usa o LLM se houver chave; senão, o modo offline.
    if API_KEY:
        try:
            return perguntar_llm(pergunta, snap, historico)
        except Exception as e:
            print("Falha no LLM, usando modo offline:", e)
    return perguntar_offline(pergunta, snap)

## 5. Testes do agente (sem interface)
Conferimos cada tipo de pergunta: a resposta, as ferramentas usadas, o gráfico correlacionado e os documentos recuperados pelo RAG.

In [ ]:
perguntas = [
    "Me dá um resumo geral do dashboard",
    "Qual a rota de coleta de hoje?",
    "Quando as lixeiras vão encher?",
    "Como está o desempenho dos coletores?",
    "Qual material mais aparece nas lixeiras?",
    "O que significa ocupação alta?",
]
for q in perguntas:
    r = perguntar_ao_ciclo(q)
    print(f"❓ {q}\n🤖 {r['resposta']}\n   ferramentas={r['ferramentas']}  gráficos={r['graficos']}  fontes={r['fontes']}  modo={r['modo']}\n")

<a id="6"></a>
## 6. Chatbot em Gradio

A interface tem duas colunas: o **chat com o Ciclo** e os **gráficos do Dashboard**. Depois de cada resposta, a aba do gráfico que o Ciclo usou é **selecionada automaticamente**, mostrando na prática a correlação entre a conversa e o Dashboard.

In [ ]:
import gradio as gr

VERDE, LARANJA, AZUL = "#2ECC71", "#E67818", "#2C3E50"
COR_MATERIAL = {"Plástico": LARANJA, "Vidro": VERDE, "Metal": "#3E5771", "Papel": "#5DADE2"}

def cor_status(oc):
    return VERDE if oc < 40 else "#F2B84B" if oc < 70 else LARANJA if oc < 90 else "#E74C3C"

# ── Os gráficos do Dashboard (mesmos dados do snapshot) ──────────────────────
def grafico_ocupacao(snap=SNAPSHOT):
    ls = sorted(snap["lixeiras"], key=lambda l: -l["ocupacao"])
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.bar([l["nome"].replace("Lixeira ", "") for l in ls], [l["ocupacao"] for l in ls],
           color=[cor_status(l["ocupacao"]) for l in ls])
    ax.axhline(90, ls="--", c="#E74C3C", lw=1); ax.set_ylim(0, 100); ax.set_ylabel("%")
    ax.set_title("Ocupação por lixeira"); plt.xticks(rotation=25, ha="right", fontsize=8); fig.tight_layout()
    return fig

def grafico_residuos(snap=SNAPSHOT):
    comp = snap["composicaoGeral"]
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    ax.pie(comp.values(), labels=comp.keys(), autopct="%1.0f%%", colors=[COR_MATERIAL[k] for k in comp],
           wedgeprops={"width": 0.4}); ax.set_title("Tipos de resíduo")
    return fig

def grafico_ranking(snap=SNAPSHOT):
    r = snap["ranking"][::-1]
    fig, ax = plt.subplots(figsize=(5, 2.8))
    ax.barh([c["nome"] for c in r], [c["coletas"] for c in r], color=VERDE)
    ax.set_title("Ranking de coletores"); ax.set_xlabel("coletas"); fig.tight_layout()
    return fig

def texto_kpis(snap=SNAPSHOT):
    k = snap["kpis"]
    return (f"### {snap['instituicao']}\n| Lixeiras | Ocupação média | Para coletar | Coletas (30 dias) |\n"
            f"|:-:|:-:|:-:|:-:|\n| **{k['lixeiras']}** | **{k['ocupacaoMedia']}%** | **{k['paraColetar']}** | **{k['coletas30d']}** |")

# gráfico do Ciclo → aba da interface
ABA_DO_GRAFICO = {"kpis": "aba_kpis", "ocupacao": "aba_ocupacao", "residuos": "aba_residuos",
                  "ranking": "aba_ranking", "atividades": "aba_kpis"}

def responder(mensagem, historico):
    # Chamado pelo Gradio a cada mensagem. 'historico' vem no formato de mensagens.
    hist_api = [{"role": m["role"], "content": m["content"]} for m in historico
                if isinstance(m.get("content"), str)]
    r = perguntar_ao_ciclo(mensagem, SNAPSHOT, hist_api)
    rodape = []
    if r["graficos"]: rodape.append("📊 gráfico: " + ", ".join(r["graficos"]))
    if r["fontes"]:   rodape.append("📚 RAG: " + ", ".join(r["fontes"]))
    resposta = r["resposta"] + ("\n\n<sub>" + " · ".join(rodape) + "</sub>" if rodape else "")
    historico = historico + [{"role": "user", "content": mensagem}, {"role": "assistant", "content": resposta}]
    aba = ABA_DO_GRAFICO.get(r["graficos"][0]) if r["graficos"] else None
    return "", historico, (gr.Tabs(selected=aba) if aba else gr.Tabs())

with gr.Blocks(title="Ciclo · G.R.U") as demo:
    gr.Markdown(f"# 🤖 Ciclo — assistente do Dashboard do G.R.U\nModo: **{MODO}**"
                + ("" if API_KEY else " (defina `CICLO_API_KEY` para usar o Gemini)"))
    with gr.Row():
        with gr.Column(scale=5):
            chat = gr.Chatbot(height=460, label="Ciclo",
                              placeholder="Pergunte sobre as lixeiras, previsões, materiais ou coletores.")
            caixa = gr.Textbox(placeholder="Ex.: Quando as lixeiras vão encher?", show_label=False)
            gr.Examples(["Me dá um resumo geral", "Qual a rota de coleta de hoje?",
                         "Quando as lixeiras vão encher?", "Qual material predomina?",
                         "Como está o desempenho dos coletores?"], inputs=caixa)
        with gr.Column(scale=4):
            with gr.Tabs() as abas:
                with gr.Tab("KPIs", id="aba_kpis"):        gr.Markdown(texto_kpis())
                with gr.Tab("Ocupação", id="aba_ocupacao"): gr.Plot(grafico_ocupacao())
                with gr.Tab("Resíduos", id="aba_residuos"): gr.Plot(grafico_residuos())
                with gr.Tab("Coletores", id="aba_ranking"):  gr.Plot(grafico_ranking())
    caixa.submit(responder, [caixa, chat], [caixa, chat, abas])

print("Interface montada. Execute a próxima célula para abrir.")

In [ ]:
# Abre a interface. No Colab, share=True gera um link público temporário.
demo.launch(share=False)

## 7. Integração com o app e o site

O mesmo agente roda no backend Node, para que a **chave da API fique só no servidor** (nunca dentro do APK ou do JavaScript do site, onde qualquer pessoa poderia extraí-la).

```
App Android / Site ──POST /ciclo/chat { pergunta, snapshot, historico }──► Backend Node
                                                                            ├─ RAG (TF-IDF)
                                                                            ├─ Agente + ferramentas
                                                                            └─ Gemini API
                   ◄── { resposta, graficos, ferramentas, fontes, modo } ──┘
```

- `snapshot`: os dados atuais dos gráficos do Dashboard.
- `graficos`: o app/site usa para **destacar o gráfico** relacionado à resposta.
- Cada gráfico do Dashboard tem um botão **"Perguntar ao Ciclo"**, que abre o chat já com uma pergunta sobre aquele gráfico.

| Arquivo | Conteúdo |
|---|---|
| `backend/src/data/ciclo-knowledge/*.md` | Base de conhecimento do RAG |
| `backend/src/services/ciclo.rag.service.js` | Chunking + TF-IDF + recuperação |
| `backend/src/services/ciclo.tools.js` | Ferramentas e previsão |
| `backend/src/services/ciclo.service.js` | Loop do agente e modo offline |
| `backend/src/routes/ciclo.route.js` | Rota `POST /ciclo/chat` |

## 8. Conclusão
- **RAG** dá ao LLM o conhecimento específico do G.R.U sem treinar o modelo.
- **Ferramentas** garantem que números e previsões venham dos dados reais, calculados em código.
- **Function calling** deixa o LLM decidir sozinho qual dado buscar para cada pergunta.
- A ligação **ferramenta → gráfico** cria a correlação entre o chatbot e o Dashboard.

**Melhorias futuras:** embeddings semânticos no RAG, previsão com mais pontos de histórico (regressão sobre várias leituras dos sensores), e respostas em streaming.